In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from radprompter import Prompt, RadPrompter, vLLMClient
from datetime import datetime
import random

In [ ]:
prompt = Prompt("../data_prep_config/prompt.toml")
prompt

In [ ]:
def generate_random_features(num_features):
    all_features = [
        "name"#, "age", "date", "hospital", "address", "phone", "email", "id", "url", "device"
    ]
    
    # Always include "name" as it's essential for a medical report
    features = ["name"]
    
    # Randomly select the remaining features
    remaining_features = random.sample(all_features[1:], min(num_features - 1, len(all_features) - 1))
    features.extend(remaining_features)
    
    # Adjust the probability of selecting "long" based on the number of features
    length_probabilities = {
        "short": 0.5,
        "medium": 0.3,
        "long": 0.2
    }
    if num_features > 5:  # If there are more than 5 features, increase the chance of selecting "long"
        length_probabilities["long"] = 0.6
        length_probabilities["short"] = 0.2
        length_probabilities["medium"] = 0.2
    elif num_features > 8:  # If there are more than 8 features, increase the chance of selecting "long" even more
        length_probabilities["long"] = 0.8
        length_probabilities["short"] = 0.1
        length_probabilities["medium"] = 0.1
    
    # Select the report length based on the adjusted probabilities
    length = random.choices(["short", "medium", "long"], weights=[length_probabilities["short"], length_probabilities["medium"], length_probabilities["long"]], k=1)[0]
    
    return {
        "report_length": length,
        "features_to_include": ", ".join(features)
    }

# Example usage
def generate_random_inputs(num_inputs):
    inputs = []
    for _ in range(num_inputs):
        num_features = random.randint(2, 11)  # Minimum 2 features, maximum 11
        inputs.append(generate_random_features(num_features))
    return inputs

In [ ]:
samples = generate_random_inputs(100)

In [ ]:
samples[-10:-5]

In [ ]:
import os
client = vLLMClient(
    model="nousresearch/hermes-3-llama-3.1-405b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0.7,
    seed=29301
)

In [ ]:
engine = RadPrompter(
    client=client,
    prompt=prompt,
    concurrency = 1,
    output_file="../data/hermes/output_18_hermes.csv",
)

engine(samples)

In [ ]:
engine.save_log(f"generation_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")